# Apriori Refined Final2

**Source script:** `apriori_refined_final2.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from run_eda_v3 import (
    SEED,
    add_moon_cyclic_features,
    build_feature_blocks,
    classify_columns,
    derive_temporal_columns,
    infer_date_column,
    infer_outcome_column,
    load_dataset,
)

ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched.csv"
DEFAULT_OUTDIR = ROOT / "outputs" / "eda_v3" / "apriori_refined_final2"
SYNODIC_MONTH_DAYS = 29.530588


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Final refined Apriori pass with stability.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT)
    parser.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR)
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--corr-threshold", type=float, default=0.85)
    parser.add_argument("--min-support", type=float, default=0.05)
    return parser.parse_args()


def normalize_name(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def sanitize_item_name(text: str) -> str:
    out = re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")
    return out or "unknown"


def format_antecedents(fs: frozenset) -> str:
    return ", ".join(sorted([str(x) for x in fs]))


def circular_distance_deg(angle_deg: np.ndarray, anchor_deg: float) -> np.ndarray:
    d = np.abs(angle_deg - anchor_deg)
    return np.minimum(d, 360.0 - d)


def correlation_filter_like_modeling(df: pd.DataFrame, weather_numeric_cols: Sequence[str], threshold: float) -> List[str]:
    if not weather_numeric_cols:
        return []
    x = df[list(weather_numeric_cols)].copy()
    x = x.apply(pd.to_numeric, errors="coerce")
    x = x.fillna(x.median(numeric_only=True))
    corr = x.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]
    return [c for c in weather_numeric_cols if c not in to_drop]


def prepare_blocks(df_raw: pd.DataFrame, corr_threshold: float) -> Tuple[pd.DataFrame, str, Dict[str, Dict[str, List[str]]]]:
    df = df_raw.copy()
    date_col = infer_date_column(df)
    outcome_col = infer_outcome_column(df)
    df, temporal_cols = derive_temporal_columns(df, date_col)
    df, _ = add_moon_cyclic_features(df)

    col_groups = classify_columns(df, outcome_col, temporal_cols)
    kept_weather_numeric = correlation_filter_like_modeling(
        df=df,
        weather_numeric_cols=col_groups["weather_model_numeric"],
        threshold=corr_threshold,
    )
    feature_blocks = build_feature_blocks(col_groups)
    env_numeric_original = set(col_groups["weather_model_numeric"])
    feature_blocks["environmental_only"]["numeric"] = sorted(kept_weather_numeric)
    feature_blocks["combined"]["numeric"] = sorted(
        [c for c in feature_blocks["combined"]["numeric"] if c not in env_numeric_original]
        + list(kept_weather_numeric)
    )
    return df, outcome_col, feature_blocks


def add_lunar_window_flags(df: pd.DataFrame, window_days: float = 2.0) -> pd.DataFrame:
    out = df.copy()
    window_deg = float(window_days) * (360.0 / SYNODIC_MONTH_DAYS)

    angle_col = None
    for c in out.columns:
        if normalize_name(c) == "moonphaseangledeg":
            angle_col = c
            break

    if angle_col is None and {"moon_phase_sin", "moon_phase_cos"}.issubset(out.columns):
        radians = np.arctan2(
            pd.to_numeric(out["moon_phase_sin"], errors="coerce").to_numpy(dtype=float),
            pd.to_numeric(out["moon_phase_cos"], errors="coerce").to_numpy(dtype=float),
        )
        angle = np.degrees(radians)
    elif angle_col is not None:
        angle = pd.to_numeric(out[angle_col], errors="coerce").to_numpy(dtype=float)
    else:
        angle = np.full(len(out), np.nan)

    angle = np.mod(angle, 360.0)
    out["moon_near_new"] = pd.Series(circular_distance_deg(angle, 0.0) <= window_deg, index=out.index).fillna(False).astype(bool)
    out["moon_near_full"] = pd.Series(circular_distance_deg(angle, 180.0) <= window_deg, index=out.index).fillna(False).astype(bool)
    return out


def build_environmental_items(df: pd.DataFrame, env_numeric_cols: Sequence[str], env_categorical_cols: Sequence[str]) -> pd.DataFrame:
    items: Dict[str, pd.Series] = {}

    for c in env_numeric_cols:
        if normalize_name(c) in {"moonphasesin", "moonphasecos"}:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        valid = s.dropna()
        if valid.empty:
            continue
        q25 = valid.quantile(0.25)
        q75 = valid.quantile(0.75)
        if pd.isna(q25) or pd.isna(q75) or float(q25) >= float(q75):
            continue
        items[f"{sanitize_item_name(c)}_low"] = (s <= q25).fillna(False).astype(bool)
        items[f"{sanitize_item_name(c)}_high"] = (s >= q75).fillna(False).astype(bool)

    for c in env_categorical_cols:
        s = df[c].astype(str).str.strip().replace({"": "missing", "nan": "missing", "None": "missing", "NaN": "missing"})
        s = s.fillna("missing")
        for v in sorted(s.unique().tolist()):
            items[f"{sanitize_item_name(c)}__{sanitize_item_name(v)}"] = (s == v).astype(bool)

    for moon_col in ["moon_near_new", "moon_near_full"]:
        if moon_col in df.columns:
            items[moon_col] = df[moon_col].fillna(False).astype(bool)

    tx = pd.DataFrame(items, index=df.index).fillna(False).astype(bool)
    return tx


def find_col(df: pd.DataFrame, patterns: Sequence[str], numeric_only: bool = False) -> Optional[str]:
    cols = [c for c in df.columns if (pd.api.types.is_numeric_dtype(df[c]) if numeric_only else True)]
    for c in cols:
        if "__missing" in c.lower():
            continue
        n = normalize_name(c)
        if any(p in n for p in patterns):
            return c
    return None


CLINICAL_SPECS: Dict[str, Dict[str, object]] = {
    "hematocrit": {"num_patterns": ["initialhematocrit", "hematocrit"], "interp_patterns": ["hematocritinterpret"], "bins": [30.3, 37.7, 45.0, 52.4]},
    "leukocyte": {"num_patterns": ["leukocytecount"], "interp_patterns": ["leukocytecountinterp"], "bins": [2.87, 7.58, 12.28, 17.03]},
    "neutrophil": {"num_patterns": ["neutrophilcount"], "interp_patterns": ["neutrophilcountinterpret"], "bins": [1.48, 4.42, 7.35, 10.3]},
    "lymphocyte": {"num_patterns": ["lymphocytecount"], "interp_patterns": ["lymphocytecountinterp"], "bins": [0.92, 2.92, 5.0, 6.89]},
    "total_protein": {"num_patterns": ["initialtotalproteinconcentration", "totalprotein"], "interp_patterns": ["initialtotalproteininterp"], "bins": [57.0, 65.0, 72.0, 79.0]},
    "albumin": {"num_patterns": ["albuminconcentration", "albumin"], "interp_patterns": ["albumininterpret"], "bins": [23.0, 28.0, 33.0, 36.0]},
    "bun": {"num_patterns": ["bunconcentration", "bun"], "interp_patterns": ["buninterp"], "bins": [6.28, 8.1, 9.92, 11.72]},
    "creatinine": {"num_patterns": ["creaconcentration", "crea", "creatinine"], "interp_patterns": ["creainterpret"], "bins": [71.0, 101.0, 131.0, 160.0]},
    "serum_k": {"num_patterns": ["kvalue", "serumk"], "interp_patterns": ["kinterp"], "bins": [3.4, 3.9, 4.3, 4.7]},
    "serum_na": {"num_patterns": ["navalue", "serumna"], "interp_patterns": ["nainterp"], "bins": [147.0, 151.0, 154.0, 157.0]},
    "serum_cl": {"num_patterns": ["clvalue", "serumcl"], "interp_patterns": ["clinterp"], "bins": [107.0, 112.0, 117.0, 121.0]},
    "pancreatic_lipase": {"num_patterns": ["pancreaticlipase", "lipase", "fpl"], "interp_patterns": [], "bins": [3.5, 3.6, 5.4], "kind": "lipase"},
    "t4": {"num_patterns": ["t4"], "interp_patterns": [], "bins": [9.0, 10.0, 30.0, 61.0], "kind": "t4"},
}


def map_numeric_state(v: float, spec: Dict[str, object]) -> str:
    if pd.isna(v):
        return "missing"
    kind = spec.get("kind", "four")
    b = spec["bins"]
    if kind == "lipase":
        if float(v) >= float(b[2]):
            return "likely_pancreatitis"
        if float(v) >= float(b[1]):
            return "possible_pancreatitis"
        return "normal"
    if kind == "t4":
        if float(v) >= float(b[3]):
            return "possible_hyperthyreosis"
        if float(v) >= float(b[2]):
            return "grey_range"
        if float(v) >= float(b[1]):
            return "normal"
        return "subnormal"
    if float(v) >= float(b[3]):
        return "increased"
    if float(v) >= float(b[2]):
        return "upper_limit"
    if float(v) >= float(b[1]):
        return "physiological"
    return "lower_limit"


def map_interp_state(text: object) -> str:
    t = normalize_name(text)
    if t in {"", "nan", "missing", "none"}:
        return "missing"
    if "likelypancreatitis" in t:
        return "likely_pancreatitis"
    if "possiblepancreatitis" in t:
        return "possible_pancreatitis"
    if "possiblehyperthyreose" in t or "possiblehyperthyreosis" in t:
        return "possible_hyperthyreosis"
    if "greyrange" in t:
        return "grey_range"
    if t in {"normal", "physiological", "physiologic"}:
        return "physiological"
    if "increased" in t or "high" in t or "erhoeht" in t:
        return "increased"
    if "upperlimit" in t:
        return "upper_limit"
    if "lowerlimit" in t or "subnormal" in t or "low" in t:
        return "lower_limit"
    return "missing"


def build_severity_consequents(df: pd.DataFrame, min_support: float) -> Tuple[pd.DataFrame, Dict[str, float], pd.DataFrame]:
    out_flags: Dict[str, pd.Series] = {}
    baselines: Dict[str, float] = {}
    state_rows: List[Dict[str, object]] = []

    for lab, spec in CLINICAL_SPECS.items():
        num_col = find_col(df, spec["num_patterns"], numeric_only=True)
        interp_col = find_col(df, spec["interp_patterns"], numeric_only=False) if spec["interp_patterns"] else None

        numeric_state = None
        if num_col is not None:
            numeric_state = pd.to_numeric(df[num_col], errors="coerce").map(lambda v: map_numeric_state(v, spec))

        interp_state = None
        if interp_col is not None:
            interp_state = df[interp_col].map(map_interp_state)

        if numeric_state is None and interp_state is None:
            continue
        if numeric_state is None:
            state = interp_state.fillna("missing")
        elif interp_state is None:
            state = numeric_state.fillna("missing")
        else:
            state = interp_state.where(interp_state != "missing", numeric_state).fillna("missing")

        for st in sorted(state.dropna().astype(str).unique().tolist()):
            if st == "missing":
                continue
            key = f"{lab}_{st}".upper()
            flag = state.astype(str).eq(st)
            prevalence = float(flag.mean())
            if prevalence < min_support or prevalence > 0.95:
                continue
            out_flags[key] = flag.astype(bool)
            baselines[key] = prevalence
            state_rows.append(
                {
                    "lab": lab,
                    "state": st,
                    "consequent_name": key,
                    "prevalence": prevalence,
                    "numeric_source": num_col if num_col else "",
                    "interpretation_source": interp_col if interp_col else "",
                }
            )


    proxy_keys = [
        "BUN_INCREASED",
        "CREATININE_INCREASED",
        "HEMATOCRIT_INCREASED",
        "TOTAL_PROTEIN_INCREASED",
        "ALBUMIN_LOWER_LIMIT",
    ]
    existing_proxy = [k for k in proxy_keys if k in out_flags]
    if existing_proxy:
        proxy = pd.concat([out_flags[k] for k in existing_proxy], axis=1).any(axis=1)
        prevalence = float(proxy.mean())
        if min_support <= prevalence <= 0.95:
            out_flags["DEHYDRATION_PROXY"] = proxy.astype(bool)
            baselines["DEHYDRATION_PROXY"] = prevalence
            state_rows.append(
                {
                    "lab": "derived",
                    "state": "dehydration_proxy",
                    "consequent_name": "DEHYDRATION_PROXY",
                    "prevalence": prevalence,
                    "numeric_source": ";".join(existing_proxy),
                    "interpretation_source": "",
                }
            )

    sev_df = pd.DataFrame(out_flags, index=df.index) if out_flags else pd.DataFrame(index=df.index)
    sev_meta = pd.DataFrame(state_rows).sort_values(["lab", "consequent_name"]).reset_index(drop=True) if state_rows else pd.DataFrame()
    return sev_df, baselines, sev_meta


def build_diagnosis_consequents(df: pd.DataFrame, outcome_col: str) -> Tuple[pd.DataFrame, Dict[str, float]]:
    y = df[outcome_col].astype(str)
    out: Dict[str, pd.Series] = {}
    baselines: Dict[str, float] = {}
    for label in sorted(y.unique().tolist()):
        key = f"diagnosis={label}"
        flag = y.eq(label)
        out[key] = flag.astype(bool)
        baselines[key] = float(flag.mean())
    return pd.DataFrame(out, index=df.index), baselines


def build_scores(df: pd.DataFrame, numeric_cols: Sequence[str], n_components: int = 6) -> np.ndarray:
    x = df[list(numeric_cols)].copy().apply(pd.to_numeric, errors="coerce")
    x = x.fillna(x.median(numeric_only=True))
    xs = StandardScaler().fit_transform(x.values)
    n_comp = int(min(max(2, n_components), xs.shape[1], xs.shape[0]))
    pca = PCA(n_components=n_comp, random_state=SEED)
    return pca.fit_transform(xs)


def canonicalize_labels(labels: np.ndarray, centers: np.ndarray) -> np.ndarray:
    order = np.argsort(centers[:, 0])
    mapping = {old: new for new, old in enumerate(order)}
    return np.array([mapping[int(x)] for x in labels], dtype=int)


def fit_kmeans_labels(scores: np.ndarray, k: int, seed: int) -> np.ndarray:
    model = KMeans(n_clusters=k, n_init=50, random_state=seed)
    raw = model.fit_predict(scores)
    return canonicalize_labels(raw, model.cluster_centers_)


def choose_kbest(scores: np.ndarray, seed: int) -> int:
    best_k = 2
    best_sil = -np.inf
    for k in range(2, 9):
        if k >= scores.shape[0]:
            continue
        labels = fit_kmeans_labels(scores, k=k, seed=seed)
        sil = float(silhouette_score(scores, labels))
        if sil > best_sil:
            best_sil = sil
            best_k = k
    return int(best_k)


@dataclass
class ClusterConsequents:
    kbest_df: pd.DataFrame
    kbest_baseline: Dict[str, float]
    k6_df: pd.DataFrame
    k6_baseline: Dict[str, float]
    kbest_meta: Dict[str, int]


def build_cluster_consequents(
    df: pd.DataFrame,
    feature_blocks: Dict[str, Dict[str, List[str]]],
    seed: int,
    fixed_kbest: Optional[Dict[str, int]] = None,
) -> ClusterConsequents:
    clin_scores = build_scores(df, feature_blocks["clinical_only"]["numeric"], n_components=6)
    comb_scores = build_scores(df, feature_blocks["combined"]["numeric"], n_components=6)

    if fixed_kbest is None:
        kbest_clin = choose_kbest(clin_scores, seed=seed)
        kbest_comb = choose_kbest(comb_scores, seed=seed)
    else:
        kbest_clin = int(fixed_kbest["clinical_only"])
        kbest_comb = int(fixed_kbest["combined"])

    labels_clin_kbest = fit_kmeans_labels(clin_scores, k=kbest_clin, seed=seed)
    labels_comb_kbest = fit_kmeans_labels(comb_scores, k=kbest_comb, seed=seed)
    labels_clin_k6 = fit_kmeans_labels(clin_scores, k=6, seed=seed) if len(df) >= 6 else np.zeros(len(df), dtype=int)
    labels_comb_k6 = fit_kmeans_labels(comb_scores, k=6, seed=seed) if len(df) >= 6 else np.zeros(len(df), dtype=int)

    kbest_items: Dict[str, pd.Series] = {}
    kbest_base: Dict[str, float] = {}
    for cl in sorted(np.unique(labels_clin_kbest).tolist()):
        name = f"cluster_clinical_kbest={int(cl)}"
        flag = pd.Series(labels_clin_kbest == cl, index=df.index).astype(bool)
        kbest_items[name] = flag
        kbest_base[name] = float(flag.mean())
    for cl in sorted(np.unique(labels_comb_kbest).tolist()):
        name = f"cluster_combined_kbest={int(cl)}"
        flag = pd.Series(labels_comb_kbest == cl, index=df.index).astype(bool)
        kbest_items[name] = flag
        kbest_base[name] = float(flag.mean())

    k6_items: Dict[str, pd.Series] = {}
    k6_base: Dict[str, float] = {}
    for cl in sorted(np.unique(labels_clin_k6).tolist()):
        name = f"cluster_clinical_k6={int(cl)}"
        flag = pd.Series(labels_clin_k6 == cl, index=df.index).astype(bool)
        k6_items[name] = flag
        k6_base[name] = float(flag.mean())
    for cl in sorted(np.unique(labels_comb_k6).tolist()):
        name = f"cluster_combined_k6={int(cl)}"
        flag = pd.Series(labels_comb_k6 == cl, index=df.index).astype(bool)
        k6_items[name] = flag
        k6_base[name] = float(flag.mean())

    return ClusterConsequents(
        kbest_df=pd.DataFrame(kbest_items, index=df.index),
        kbest_baseline=kbest_base,
        k6_df=pd.DataFrame(k6_items, index=df.index),
        k6_baseline=k6_base,
        kbest_meta={"clinical_only": int(kbest_clin), "combined": int(kbest_comb)},
    )


def mine_rules_for_consequents(
    tx_env: pd.DataFrame,
    consequent_df: pd.DataFrame,
    baseline_map: Dict[str, float],
    track: str,
    min_support: float,
    min_lift: float,
    singleton_min_lift: float,
) -> pd.DataFrame:
    if consequent_df.empty or tx_env.empty:
        return pd.DataFrame(
            columns=[
                "track",
                "consequent_name",
                "antecedents",
                "support",
                "support_count",
                "confidence",
                "lift",
                "baseline_prevalence",
                "leverage",
                "conviction",
            ]
        )

    n_rows = len(tx_env)
    antecedent_cols = tx_env.columns.tolist()
    rows: List[pd.DataFrame] = []
    for consequent in consequent_df.columns.tolist():
        baseline = float(baseline_map.get(consequent, np.nan))
        if not np.isfinite(baseline) or baseline <= 0:
            continue
        min_conf = baseline
        data = pd.concat([tx_env, consequent_df[[consequent]]], axis=1).fillna(False).astype(bool)
        itemsets = apriori(data, min_support=min_support, use_colnames=True, max_len=4)
        if itemsets.empty:
            continue
        rules = association_rules(itemsets, metric="confidence", min_threshold=min_conf)
        if rules.empty:
            continue

        rules = rules[
            rules["antecedents"].apply(lambda s: 1 <= len(s) <= 3 and set(s).issubset(set(antecedent_cols)))
            & rules["consequents"].apply(lambda s: len(s) == 1 and consequent in s)
            & (rules["support"] >= min_support)
            & (rules["confidence"] >= min_conf)
            & (rules["lift"] >= min_lift)
        ].copy()
        if rules.empty:
            continue
        rules["antecedent_len"] = rules["antecedents"].apply(len)
        rules = rules[~((rules["antecedent_len"] == 1) & (rules["lift"] < singleton_min_lift))].copy()
        if rules.empty:
            continue

        rules["track"] = track
        rules["consequent_name"] = consequent
        rules["antecedents"] = rules["antecedents"].apply(format_antecedents)
        rules["support_count"] = (rules["support"] * n_rows).round().astype(int)
        rules["baseline_prevalence"] = baseline
        keep = ["track", "consequent_name", "antecedents", "support", "support_count", "confidence", "lift", "baseline_prevalence"]
        if "leverage" in rules.columns:
            keep.append("leverage")
        if "conviction" in rules.columns:
            keep.append("conviction")
        rows.append(rules[keep].reset_index(drop=True))

    if not rows:
        return pd.DataFrame(
            columns=[
                "track",
                "consequent_name",
                "antecedents",
                "support",
                "support_count",
                "confidence",
                "lift",
                "baseline_prevalence",
                "leverage",
                "conviction",
            ]
        )
    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values(["lift", "confidence", "support"], ascending=[False, False, False]).reset_index(drop=True)
    return out


def _rule_key(df: pd.DataFrame) -> List[Tuple[str, str]]:
    if df.empty:
        return []
    return list(zip(df["consequent_name"].astype(str), df["antecedents"].astype(str)))


def build_stable_rules(full_rules: pd.DataFrame, fold_counter: Counter, min_folds: int = 3) -> pd.DataFrame:
    if full_rules.empty:
        return full_rules.copy()
    out = full_rules.copy()
    out["fold_hits"] = [int(fold_counter.get((str(r["consequent_name"]), str(r["antecedents"])), 0)) for _, r in out.iterrows()]
    out = out[out["fold_hits"] >= min_folds].copy()
    out = out.sort_values(["fold_hits", "lift", "confidence"], ascending=[False, False, False]).reset_index(drop=True)
    return out


def top5_md(df: pd.DataFrame) -> str:
    if df.empty:
        return "_No rules met thresholds._"
    sub = df.sort_values(["lift", "confidence"], ascending=[False, False]).head(5).copy()
    cols = ["consequent_name", "antecedents", "support", "confidence", "lift"]
    lines = ["| " + " | ".join(cols) + " |", "| --- | --- | --- | --- | --- |"]
    for _, r in sub.iterrows():
        lines.append(
            f"| {r['consequent_name']} | {r['antecedents']} | {r['support']:.3f} | {r['confidence']:.3f} | {r['lift']:.3f} |"
        )
    return "\n".join(lines)


def run_fold_stability(
    df_base: pd.DataFrame,
    outcome_col: str,
    feature_blocks: Dict[str, Dict[str, List[str]]],
    seed: int,
    min_support: float,
    kbest_map: Dict[str, int],
) -> Dict[str, Counter]:
    y = df_base[outcome_col].astype(str)
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    counters = {"diagnosis": Counter(), "severity": Counter(), "cluster_kbest": Counter(), "cluster_k6": Counter()}

    for train_idx, _ in splitter.split(df_base, y):
        fold_df = df_base.iloc[train_idx].copy()
        fold_df = add_lunar_window_flags(fold_df, window_days=2.0)
        tx_env = build_environmental_items(
            fold_df,
            env_numeric_cols=feature_blocks["environmental_only"]["numeric"],
            env_categorical_cols=feature_blocks["environmental_only"]["categorical"],
        )
        diag_df, diag_base = build_diagnosis_consequents(fold_df, outcome_col)
        sev_df, sev_base, _ = build_severity_consequents(fold_df, min_support=min_support)
        cluster = build_cluster_consequents(
            fold_df,
            feature_blocks=feature_blocks,
            seed=seed,
            fixed_kbest=kbest_map,
        )

        rules_diag = mine_rules_for_consequents(tx_env, diag_df, diag_base, "diagnosis", min_support, 1.25, 1.5)
        rules_sev = mine_rules_for_consequents(tx_env, sev_df, sev_base, "severity", min_support, 1.20, 1.20)
        rules_kbest = mine_rules_for_consequents(tx_env, cluster.kbest_df, cluster.kbest_baseline, "cluster_kbest", min_support, 1.20, 1.20)
        rules_k6 = mine_rules_for_consequents(tx_env, cluster.k6_df, cluster.k6_baseline, "cluster_k6", min_support, 1.20, 1.20)

        for k in _rule_key(rules_diag):
            counters["diagnosis"][k] += 1
        for k in _rule_key(rules_sev):
            counters["severity"][k] += 1
        for k in _rule_key(rules_kbest):
            counters["cluster_kbest"][k] += 1
        for k in _rule_key(rules_k6):
            counters["cluster_k6"][k] += 1

    return counters


def strongest_rule(df: pd.DataFrame) -> Optional[pd.Series]:
    if df.empty:
        return None
    return df.sort_values(["lift", "confidence", "support"], ascending=[False, False, False]).iloc[0]


def write_summary(
    out_path: Path,
    env_item_count: int,
    kbest_map: Dict[str, int],
    k6_reporting_passed: bool,
    diagnosis_rules: pd.DataFrame,
    severity_rules: pd.DataFrame,
    cluster_kbest_rules: pd.DataFrame,
    cluster_k6_rules: pd.DataFrame,
) -> None:
    lines: List[str] = []
    lines.append("# Apriori Refined Final 2 Summary")
    lines.append("")
    lines.append("## Configuration")
    lines.append("- Antecedents: environmental-only items (LOW/HIGH at 25th/75th quantiles + environmental categorical one-hot + moon_near_new/moon_near_full).")
    lines.append("- Itemset max length = 4; antecedent size <= 3.")
    lines.append("- min_support = 0.05.")
    lines.append("- min_confidence = consequent baseline prevalence (dynamic).")
    lines.append("- lift thresholds: diagnosis >= 1.25; severity/cluster >= 1.20.")
    lines.append("- Singleton diagnosis rules shown only if lift >= 1.5.")
    lines.append("- Stability screen: 5 stratified folds by diagnosis, stable if appears in >=3/5 training folds.")
    lines.append(f"- Environmental antecedent item count: {env_item_count}.")
    lines.append(f"- k_best used for cluster consequents: clinical_only={kbest_map['clinical_only']}, combined={kbest_map['combined']}.")
    lines.append("")
    lines.append("## Reporting Policy")
    lines.append("- k_best cluster rules reported fully.")
    if k6_reporting_passed:
        lines.append("- k=6 cluster rules met reporting threshold (>=3 rules with lift >= 1.3).")
    else:
        lines.append("- k=6 did not meet reporting threshold.")
    lines.append("")
    lines.append("## Top 5 Rules By Track (Lift)")
    lines.append("### Diagnosis")
    lines.append(top5_md(diagnosis_rules))
    lines.append("")
    lines.append("### Severity")
    lines.append(top5_md(severity_rules))
    lines.append("")
    lines.append("### Cluster k_best")
    lines.append(top5_md(cluster_kbest_rules))
    lines.append("")
    lines.append("### Cluster k=6")
    lines.append(top5_md(cluster_k6_rules if k6_reporting_passed else pd.DataFrame()))
    lines.append("")
    lines.append("## Note")
    lines.append("- Exploratory association mining only; non-causal and hypothesis-generating.")
    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def main() -> None:
    args = parse_args()
    np.random.seed(args.seed)
    args.outdir.mkdir(parents=True, exist_ok=True)

    df_raw = load_dataset(args.input)
    drop_weather_reason = [c for c in df_raw.columns if "weather_missing_reason" in c.lower()]
    if drop_weather_reason:
        df_raw = df_raw.drop(columns=drop_weather_reason)

    df, outcome_col, feature_blocks = prepare_blocks(df_raw, corr_threshold=args.corr_threshold)
    df = add_lunar_window_flags(df, window_days=2.0)

    tx_env = build_environmental_items(
        df=df,
        env_numeric_cols=feature_blocks["environmental_only"]["numeric"],
        env_categorical_cols=feature_blocks["environmental_only"]["categorical"],
    )
    diagnosis_df, diagnosis_base = build_diagnosis_consequents(df, outcome_col)
    severity_df, severity_base, severity_meta = build_severity_consequents(df, min_support=args.min_support)
    cluster = build_cluster_consequents(df, feature_blocks=feature_blocks, seed=args.seed)
    kbest_map = cluster.kbest_meta

    diagnosis_rules = mine_rules_for_consequents(
        tx_env, diagnosis_df, diagnosis_base, "diagnosis", args.min_support, 1.25, 1.5
    )
    severity_rules = mine_rules_for_consequents(
        tx_env, severity_df, severity_base, "severity", args.min_support, 1.20, 1.20
    )
    cluster_kbest_rules = mine_rules_for_consequents(
        tx_env, cluster.kbest_df, cluster.kbest_baseline, "cluster_kbest", args.min_support, 1.20, 1.20
    )
    cluster_k6_rules = mine_rules_for_consequents(
        tx_env, cluster.k6_df, cluster.k6_baseline, "cluster_k6", args.min_support, 1.20, 1.20
    )


    k6_reporting_passed = int((cluster_k6_rules["lift"] >= 1.3).sum()) >= 3 if not cluster_k6_rules.empty else False


    fold_counters = run_fold_stability(
        df_base=df,
        outcome_col=outcome_col,
        feature_blocks=feature_blocks,
        seed=args.seed,
        min_support=args.min_support,
        kbest_map=kbest_map,
    )
    diagnosis_stable = build_stable_rules(diagnosis_rules, fold_counters["diagnosis"], min_folds=3)
    severity_stable = build_stable_rules(severity_rules, fold_counters["severity"], min_folds=3)
    cluster_kbest_stable = build_stable_rules(cluster_kbest_rules, fold_counters["cluster_kbest"], min_folds=3)
    cluster_k6_stable = build_stable_rules(cluster_k6_rules, fold_counters["cluster_k6"], min_folds=3)


    diagnosis_rules.to_csv(args.outdir / "diagnosis_rules_all.csv", index=False)
    diagnosis_stable.to_csv(args.outdir / "diagnosis_rules_stable.csv", index=False)
    severity_rules.to_csv(args.outdir / "severity_rules_all.csv", index=False)
    severity_stable.to_csv(args.outdir / "severity_rules_stable.csv", index=False)
    cluster_kbest_rules.to_csv(args.outdir / "cluster_rules_kbest_all.csv", index=False)
    cluster_kbest_stable.to_csv(args.outdir / "cluster_rules_kbest_stable.csv", index=False)
    severity_meta.to_csv(args.outdir / "severity_state_harmonization_map.csv", index=False)

    if k6_reporting_passed:
        cluster_k6_rules.to_csv(args.outdir / "cluster_rules_k6_all.csv", index=False)
        cluster_k6_stable.to_csv(args.outdir / "cluster_rules_k6_stable.csv", index=False)

    write_summary(
        out_path=args.outdir / "apriori_refined_final2_summary.md",
        env_item_count=tx_env.shape[1],
        kbest_map=kbest_map,
        k6_reporting_passed=k6_reporting_passed,
        diagnosis_rules=diagnosis_rules,
        severity_rules=severity_rules,
        cluster_kbest_rules=cluster_kbest_rules,
        cluster_k6_rules=cluster_k6_rules,
    )


    print("=== Apriori Refined Final2 ===")
    print(f"Output directory: {args.outdir}")
    tracks = {
        "diagnosis": diagnosis_rules,
        "severity": severity_rules,
        "cluster_kbest": cluster_kbest_rules,
        "cluster_k6": cluster_k6_rules if k6_reporting_passed else pd.DataFrame(),
    }
    for name, rdf in tracks.items():
        top = strongest_rule(rdf)
        if top is None:
            print(f"{name}: strongest => none")
        else:
            print(
                f"{name}: strongest => {top['consequent_name']} <= {top['antecedents']} "
                f"(lift={top['lift']:.3f}, conf={top['confidence']:.3f}, support={top['support']:.3f})"
            )

    print("Stable rule counts:")
    print(f"- diagnosis: {len(diagnosis_stable)}")
    print(f"- severity: {len(severity_stable)}")
    print(f"- cluster_kbest: {len(cluster_kbest_stable)}")
    print(f"- cluster_k6: {len(cluster_k6_stable) if k6_reporting_passed else 0}")
    print(f"k=6 reporting condition passed: {'yes' if k6_reporting_passed else 'no'}")


if __name__ == "__main__":
    main()
